# Lazer Parametreleri Optimizasyonu — 1.65 W Analizi

**Deney Stratejisi:**
- Power = 1.65 W (sabit)
- Pass = 1, 5, 10
- Phase 1: Mevcut tüm hız verileri
- Phase 2: Yapılandırılmış hızlar → 10, 100, 200, …, 3000 mm/s

In [ ]:
# ─── KURULUM ───────────────────────────────────────────────────────────────
# Gerekli paketleri yükle (Colab'da zaten çoğu var)
# !pip install -q scikit-learn pandas matplotlib seaborn

In [ ]:
# ─── IMPORT ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneOut, cross_val_score, KFold
import warnings
warnings.filterwarnings('ignore')

# Görsel ayarlar
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PASS_COLORS = {1: '#2196F3', 5: '#FF9800', 10: '#4CAF50'}
PASS_MARKERS = {1: 'o', 5: 's', 10: '^'}

print('Tüm paketler yüklendi.')

## 1. Veri Yükleme

CSV dosyasını Colab'a yüklemek için aşağıdaki hücreyi çalıştırın.

In [ ]:
# ─── VERİ YÜKLEME ──────────────────────────────────────────────────────────
# SEÇENEK A: Dosyayı manuel yükle (küçük CSV için)
from google.colab import files
uploaded = files.upload()          # qfactors_1.0um.csv seçin
CSV_NAME = list(uploaded.keys())[0]

# SEÇENEK B: Google Drive'dan yükle (büyük CSV için)
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_NAME = '/content/drive/MyDrive/YOUR_FOLDER/qfactors_1.0um.csv'

df_raw = pd.read_csv(CSV_NAME)
print(f'Ham veri: {df_raw.shape[0]:,} satır, {df_raw.shape[1]} sütun')
df_raw.head(3)

## 2. Filtreleme ve Ön İşleme

In [ ]:
# ─── RENAME + TEMEL FİLTRE ─────────────────────────────────────────────────
RENAME = {
    'Power(W)':              'power',
    'Scanning speed(mm/s)':  'speed',
    'Pass':                  'n_pass',
    'q_factor':              'q_factor',
}
df = df_raw.rename(columns=RENAME).copy()

# Deney parametreleri
POWER   = 1.65
PASSES  = [1, 5, 10]

# Phase 1: Power=1.65, Pass=[1,5,10], geçerli q_factor, tüm hızlar
mask = (
    (df['power'] == POWER) &
    (df['n_pass'].isin(PASSES)) &
    df['q_factor'].notna() &
    df['q_factor'].between(0, 1)
)
df_p1 = df.loc[mask].copy()

# Phase 2: Yapılandırılmış hızlar (10, 100, 200, …, 3000)
STRUCTURED_SPEEDS = [10] + list(range(100, 3001, 100))
df_p2 = df_p1[df_p1['speed'].isin(STRUCTURED_SPEEDS)].copy()

print(f'Phase 1 — tüm hızlar:           {len(df_p1):,} satır')
print(f'Phase 2 — yapılandırılmış hızlar: {len(df_p2):,} satır')
print()
print('Phase 1 hız değerleri:', sorted(df_p1['speed'].unique()))
print('Phase 2 hız değerleri:', sorted(df_p2['speed'].unique()))

In [ ]:
# ─── GÖRÜNTÜ BAZINDA AGREGELEŞTİRME (medyan) ──────────────────────────────
# Her görüntü dosyası için bir satır: q_factor, A, R → medyan

def aggregate_per_image(df):
    """Her filename için medyan A, R, Q değerlerini alır."""
    agg = (
        df.groupby('filename', as_index=False)
        .agg(
            speed   = ('speed',    'first'),
            n_pass  = ('n_pass',   'first'),
            power   = ('power',    'first'),
            Q       = ('q_factor', 'median'),
            A       = ('A',        'median'),
            R       = ('R',        'median'),
            n_slices= ('q_factor', 'count'),
        )
    )
    return agg.sort_values(['n_pass', 'speed']).reset_index(drop=True)


agg_p1 = aggregate_per_image(df_p1)   # Phase 1
agg_p2 = aggregate_per_image(df_p2)   # Phase 2

print('Phase 1 — görüntü sayısı:', len(agg_p1))
print('Phase 2 — görüntü sayısı:', len(agg_p2))
print()

# Pass bazında dağılım
for phase, agg in [('Phase 1', agg_p1), ('Phase 2', agg_p2)]:
    print(f'{phase}:')
    print(agg.groupby('n_pass').agg(n_images=('filename','count'),
                                    speed_min=('speed','min'),
                                    speed_max=('speed','max')).to_string())
    print()

# CSV olarak kaydet (sonraki oturumlarda tekrar yüklemek zorunda kalmamak için)
agg_p1.to_csv('laser_165W_phase1.csv', index=False)
agg_p2.to_csv('laser_165W_phase2.csv', index=False)
print('Filtrelenmiş veriler kaydedildi: laser_165W_phase1.csv / phase2.csv')

## 3. Veri Analizi — Görselleştirme

### 3a. Hız vs Q-Factor (3 Pass Karşılaştırması)

In [ ]:
# ─── ANA GRAFİK: Hız vs Q-Factor, Pass'a göre renklendirilmiş ─────────────
# Hangi aşamayı görselleştireceğinizi seçin:
PLOT_DATA = agg_p1   # Phase 1 için agg_p1, Phase 2 için agg_p2
PHASE_LABEL = 'Phase 1 — Tüm Hızlar'   # grafik başlığı

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(f'1.65 W — Hız vs Q-Factor  |  {PHASE_LABEL}', fontsize=13, fontweight='bold')

for ax, n_pass in zip(axes, PASSES):
    sub = PLOT_DATA[PLOT_DATA['n_pass'] == n_pass].sort_values('speed')
    color = PASS_COLORS[n_pass]
    marker = PASS_MARKERS[n_pass]

    ax.scatter(sub['speed'], sub['Q'], color=color, marker=marker,
               s=55, zorder=3, label=f'{n_pass} pass')

    # LOWESS benzeri: hareketli medyan (pencere=7)
    if len(sub) >= 7:
        rolling_med = sub.set_index('speed')['Q'].rolling(7, center=True, min_periods=3).median()
        ax.plot(rolling_med.index, rolling_med.values,
                color=color, lw=1.8, alpha=0.7, label='Hareketli Medyan')

    ax.set_title(f'{n_pass} Pass', fontsize=12)
    ax.set_xlabel('Hız (mm/s)')
    if ax == axes[0]:
        ax.set_ylabel('Q-Factor')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot_speed_vs_q_per_pass.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: plot_speed_vs_q_per_pass.png')

In [ ]:
# ─── OVERLAY GRAFİĞİ: 3 Pass Aynı Eksende ────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle(f'1.65 W — Hız vs Q-Factor  |  {PHASE_LABEL}', fontsize=13, fontweight='bold')

for n_pass in PASSES:
    sub = PLOT_DATA[PLOT_DATA['n_pass'] == n_pass].sort_values('speed')
    color = PASS_COLORS[n_pass]
    marker = PASS_MARKERS[n_pass]

    ax.scatter(sub['speed'], sub['Q'], color=color, marker=marker,
               s=55, zorder=3, label=f'{n_pass} pass (veri)')

    if len(sub) >= 7:
        rolling_med = sub.set_index('speed')['Q'].rolling(7, center=True, min_periods=3).median()
        ax.plot(rolling_med.index, rolling_med.values,
                color=color, lw=2, alpha=0.75)

ax.set_xlabel('Hız (mm/s)', fontsize=12)
ax.set_ylabel('Q-Factor', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('plot_speed_vs_q_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 3 HEDEF DEĞİŞKEN: Q, A, R ────────────────────────────────────────────
targets_info = {'Q': 'Q-Factor', 'A': 'Amplitude (A)', 'R': 'Radius (R)'}

fig, axes = plt.subplots(len(targets_info), 1, figsize=(12, 4 * len(targets_info)), sharex=True)
fig.suptitle(f'1.65 W — Hız vs Hedef Değişkenler  |  {PHASE_LABEL}', fontsize=13, fontweight='bold')

for ax, (col, label) in zip(axes, targets_info.items()):
    for n_pass in PASSES:
        sub = PLOT_DATA[PLOT_DATA['n_pass'] == n_pass].sort_values('speed')
        ax.scatter(sub['speed'], sub[col],
                   color=PASS_COLORS[n_pass], marker=PASS_MARKERS[n_pass],
                   s=45, zorder=3, label=f'{n_pass} pass')
        if len(sub) >= 7:
            rm = sub.set_index('speed')[col].rolling(7, center=True, min_periods=3).median()
            ax.plot(rm.index, rm.values, color=PASS_COLORS[n_pass], lw=1.8, alpha=0.7)
    ax.set_ylabel(label, fontsize=11)
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Hız (mm/s)', fontsize=12)
plt.tight_layout()
plt.savefig('plot_all_targets.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── KORElASYON ISISI ──────────────────────────────────────────────────────
cols_for_corr = ['speed', 'n_pass', 'Q', 'A', 'R']
corr_matrix = PLOT_DATA[cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title(f'Korelasyon Matrisi  |  {PHASE_LABEL}', fontsize=11)
plt.tight_layout()
plt.savefig('plot_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. ML Modelleme

**Strateji:**
- Az ama temiz veri → Gaussian Process Regression (GPR) en uygun teorik seçim
- 4 model karşılaştırması: Polynomial Ridge, SVR, GPR, Random Forest
- Küçük veri → LOO-CV (Leave-One-Out Cross Validation)
- Hedef: Q-Factor (A ve R için aynı kodu tekrarlayabilirsiniz)

In [ ]:
# ─── MODEL KARŞILAŞTIRMA FONKSİYONU ───────────────────────────────────────
# Hangi veriyi ve hedefi kullanacağınızı seçin:
MODEL_DATA   = agg_p2   # Phase 2 (yapılandırılmış, daha temiz)
TARGET_COL   = 'Q'      # 'Q', 'A', veya 'R'
FEATURES     = ['speed', 'n_pass']

X_full = MODEL_DATA[FEATURES].values
y_full = MODEL_DATA[TARGET_COL].values

print(f'Model verisi: {len(X_full)} örnek  |  Hedef: {TARGET_COL}')
print(f'Özellikler: {FEATURES}')
print(f'y aralığı: [{y_full.min():.4f}, {y_full.max():.4f}]')

In [ ]:
# ─── 4 MODEL TANIMI ────────────────────────────────────────────────────────
# 1. Polynomial Ridge Regression (degree=3)
poly_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=3, include_bias=False)),
    ('ridge',  Ridge(alpha=1.0)),
])

# 2. SVR (RBF kernel)
svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr',    SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.01)),
])

# 3. Gaussian Process Regression — küçük fizik veri seti için ideal
kernel_gpr = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=1e-3)
gpr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('gpr',    GaussianProcessRegressor(kernel=kernel_gpr, n_restarts_optimizer=5,
                                        normalize_y=True, random_state=42)),
])

# 4. Random Forest (baseline)
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf',     RandomForestRegressor(n_estimators=300, max_depth=10,
                                     min_samples_leaf=2, random_state=42)),
])

MODELS = {
    'Poly Ridge (d=3)':  poly_pipe,
    'SVR (RBF)':         svr_pipe,
    'GPR (Matern)':      gpr_pipe,
    'Random Forest':     rf_pipe,
}

In [ ]:
# ─── LOO-CV DEĞERLENDİRME ─────────────────────────────────────────────────
# Küçük veri setinde en güvenilir performans tahmini: Leave-One-Out CV

loo = LeaveOneOut()
results = []

for name, model in MODELS.items():
    cv_scores = cross_val_score(model, X_full, y_full,
                                cv=loo, scoring='r2', n_jobs=-1)
    # MAE için ayrıca tahmin yap
    y_pred_loo = np.zeros_like(y_full, dtype=float)
    for train_idx, test_idx in loo.split(X_full):
        model.fit(X_full[train_idx], y_full[train_idx])
        y_pred_loo[test_idx] = model.predict(X_full[test_idx])

    r2_loo   = r2_score(y_full, y_pred_loo)
    mae_loo  = mean_absolute_error(y_full, y_pred_loo)
    rmse_loo = np.sqrt(mean_squared_error(y_full, y_pred_loo))

    # Eğitim skoru (fit all)
    model.fit(X_full, y_full)
    r2_train = r2_score(y_full, model.predict(X_full))

    results.append({
        'Model':       name,
        'R2 Train':    round(r2_train, 4),
        'R2 LOO-CV':   round(r2_loo, 4),
        'MAE LOO-CV':  round(mae_loo, 4),
        'RMSE LOO-CV': round(rmse_loo, 4),
        'Overfitting': round(r2_train - r2_loo, 4),
    })
    print(f'[{name}]  R2_train={r2_train:.4f}  R2_LOO={r2_loo:.4f}  '
          f'MAE={mae_loo:.4f}  RMSE={rmse_loo:.4f}')

df_results = pd.DataFrame(results).sort_values('R2 LOO-CV', ascending=False)
print('\n=== MODEL KARŞILAŞTIRMA TABLOSU ===')
print(df_results.to_string(index=False))
df_results.to_csv('model_results.csv', index=False)

In [ ]:
# ─── MODEL KARŞILAŞTIRMA BAR GRAFİĞİ ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Model Karşılaştırma — LOO-CV  |  Hedef: {TARGET_COL}',
             fontsize=12, fontweight='bold')

model_names = df_results['Model'].tolist()
x = np.arange(len(model_names))
palette = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

# R2 karşılaştırması
ax = axes[0]
r2_train_vals = [df_results.loc[df_results['Model']==m, 'R2 Train'].values[0] for m in model_names]
r2_loo_vals   = [df_results.loc[df_results['Model']==m, 'R2 LOO-CV'].values[0] for m in model_names]
bars_train = ax.bar(x - 0.2, r2_train_vals, 0.35, label='Train R2', color='steelblue', alpha=0.85)
bars_loo   = ax.bar(x + 0.2, r2_loo_vals,   0.35, label='LOO-CV R2', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('R2')
ax.set_title('R2 (Train vs LOO-CV)')
ax.legend()
ax.set_ylim(-0.1, 1.05)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')

# MAE karşılaştırması
ax = axes[1]
mae_vals = [df_results.loc[df_results['Model']==m, 'MAE LOO-CV'].values[0] for m in model_names]
bars = ax.bar(x, mae_vals, 0.5, color=palette[:len(model_names)], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('MAE')
ax.set_title('MAE (LOO-CV) — düşük = iyi')
for bar, val in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── EN İYİ MODEL — TAHMİN vs GERÇEK ─────────────────────────────────────
best_model_name = df_results.iloc[0]['Model']
best_model      = MODELS[best_model_name]
best_model.fit(X_full, y_full)

# LOO tahminlerini yeniden hesapla
y_pred_best = np.zeros_like(y_full, dtype=float)
for train_idx, test_idx in loo.split(X_full):
    best_model.fit(X_full[train_idx], y_full[train_idx])
    y_pred_best[test_idx] = best_model.predict(X_full[test_idx])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_full, y_pred_best, alpha=0.7, s=40, color='steelblue', edgecolors='navy', lw=0.3)
lo, hi = min(y_full.min(), y_pred_best.min()), max(y_full.max(), y_pred_best.max())
margin = (hi - lo) * 0.05
ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin], 'r--', lw=1.5, label='İdeal')
r2 = r2_score(y_full, y_pred_best)
mae = mean_absolute_error(y_full, y_pred_best)
ax.set_xlabel(f'Gerçek {TARGET_COL}', fontsize=12)
ax.set_ylabel(f'Tahmin {TARGET_COL}', fontsize=12)
ax.set_title(f'{best_model_name}  |  LOO-CV  R2={r2:.4f}  MAE={mae:.4f}', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('plot_pred_vs_actual_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── GPR — BELİRSİZLİK TAHMİNİ (Confidence Band) ─────────────────────────
# GPR'ın en büyük avantajı: her tahmin için güven aralığı verir

# Tüm veri üzerinde GPR'ı eğit
gpr_pipe.fit(X_full, y_full)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(f'GPR Tahmin + Güven Aralığı  |  1.65 W  |  Hedef: {TARGET_COL}',
             fontsize=12, fontweight='bold')

speed_range = np.linspace(MODEL_DATA['speed'].min(), MODEL_DATA['speed'].max(), 300)

for ax, n_pass in zip(axes, PASSES):
    # Veri noktaları
    sub = MODEL_DATA[MODEL_DATA['n_pass'] == n_pass].sort_values('speed')
    ax.scatter(sub['speed'], sub[TARGET_COL],
               color=PASS_COLORS[n_pass], marker=PASS_MARKERS[n_pass],
               s=55, zorder=4, label='Veri')

    # GPR tahmin eğrisi
    X_pred = np.column_stack([speed_range, np.full_like(speed_range, n_pass)])

    # GPR pipeline'dan scaler'ı manuel uygula
    scaler_gpr = gpr_pipe.named_steps['scaler']
    gpr_model  = gpr_pipe.named_steps['gpr']
    X_pred_scaled = scaler_gpr.transform(X_pred)
    y_mean, y_std = gpr_model.predict(X_pred_scaled, return_std=True)

    ax.plot(speed_range, y_mean, color=PASS_COLORS[n_pass], lw=2, label='GPR Ortalama')
    ax.fill_between(speed_range, y_mean - 2*y_std, y_mean + 2*y_std,
                    alpha=0.2, color=PASS_COLORS[n_pass], label='±2σ Güven')

    ax.set_title(f'{n_pass} Pass', fontsize=12)
    ax.set_xlabel('Hız (mm/s)')
    if ax == axes[0]:
        ax.set_ylabel(TARGET_COL)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plot_gpr_confidence.png', dpi=150, bbox_inches='tight')
plt.show()
print('GPR güven bantları kaydedildi.')

In [ ]:
# ─── OPTİMAL PARAMETRE TAHMİNİ (GPR ile) ──────────────────────────────────
# Q-Factor'ü maksimize eden hız ve pass kombinasyonunu bul

gpr_pipe.fit(X_full, y_full)  # tüm veri üzerinde tekrar eğit

# Aday parametre ızgarası
speed_grid = np.arange(10, 3001, 10)
pass_grid  = [1, 5, 10]

grid_rows = []
for n_pass in pass_grid:
    X_grid = np.column_stack([speed_grid, np.full_like(speed_grid, n_pass, dtype=float)])
    X_grid_scaled = gpr_pipe.named_steps['scaler'].transform(X_grid)
    y_pred_grid, y_std_grid = gpr_pipe.named_steps['gpr'].predict(X_grid_scaled, return_std=True)
    for s, yp, ys in zip(speed_grid, y_pred_grid, y_std_grid):
        grid_rows.append({'speed': s, 'n_pass': n_pass, 'Q_pred': yp, 'Q_std': ys})

df_grid = pd.DataFrame(grid_rows)
best_idx = df_grid['Q_pred'].idxmax()
best_row = df_grid.loc[best_idx]

print('=== GPR Optimal Parametre Tahmini ===')
print(f"En iyi Q-Factor: {best_row['Q_pred']:.4f} ± {best_row['Q_std']:.4f}")
print(f"Hız:  {best_row['speed']:.0f} mm/s")
print(f"Pass: {best_row['n_pass']:.0f}")
print(f"Power: 1.65 W (sabit)")

In [ ]:
# ─── ÇIKTI DOSYALARI ───────────────────────────────────────────────────────
from google.colab import files as colab_files

output_files = [
    'laser_165W_phase1.csv',
    'laser_165W_phase2.csv',
    'model_results.csv',
    'plot_speed_vs_q_per_pass.png',
    'plot_speed_vs_q_overlay.png',
    'plot_all_targets.png',
    'plot_correlation.png',
    'plot_model_comparison.png',
    'plot_pred_vs_actual_best.png',
    'plot_gpr_confidence.png',
]

import os
for f in output_files:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'İndirildi: {f}')
    else:
        print(f'BULUNAMADI: {f}')